# 01b. Error analysis of in-context learning

Which senses does the best ICL configuration get wrong, and why? Runs on CPU from the prediction files written by
`01_icl` (no model needed). Figures go to `results/figures/`.

In [ ]:
# Setup: on Colab or Kaggle, clone (or update) the repo so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/bkadyrbay-hash/lab1-supersense-llm.git"
BRANCH = "a/icl-experiments"  # set to your own branch to test code that is not merged yet
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/lab1-supersense-llm"
    if not os.path.exists(REPO_DIR):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

from src.data import SUPER2HYPER, SUPERSENSES, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, plot_confusion

FIG_DIR = RESULTS_DIR / "figures"
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
icl_dev = metrics[(metrics["method"] == "icl") & (metrics["split"] == "rand-dev")].drop_duplicates("config_id", keep="last")
BEST = icl_dev.sort_values("macro_f1_super").iloc[-1]["config_id"]
print("Best configuration on rand-dev:", BEST)


def load_preds(config_id, split="rand-dev"):
    """Prediction file of one ICL run, with a column telling whether the supersense is correct."""
    preds = pd.read_csv(RESULTS_DIR / "predictions" / f"icl__{config_id}__{split}.csv", keep_default_na=False)
    return preds.assign(correct=preds["gold_supersense"] == preds["pred_supersense"])


preds = load_preds(BEST)
train = get_split(load_data(), "train")

## 1. Which classes are hard? Per-class F1 vs frequency in train

In [ ]:
scores = evaluate(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES)
per_class = pd.DataFrame({"f1": scores["per_class"]}).assign(
    n_dev=preds["gold_supersense"].value_counts(), n_train=train["supersense"].value_counts()).sort_values("f1")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(per_class["n_train"], per_class["f1"], s=12 + per_class["n_dev"])
for label, row in per_class.iterrows():
    ax.annotate(label, (row["n_train"], row["f1"]), fontsize=7, xytext=(3, 2), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("training instances (log scale; ICL only sees a few of them)")
ax.set_ylabel("F1 on rand-dev")
ax.set_title(f"Per-class F1 on rand-dev, {BEST}\n(dot size = number of rand-dev instances)", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "icl_per_class_f1.png", dpi=200)
plt.show()
per_class

## 2. Confusions

In [ ]:
plot_confusion(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES,
               FIG_DIR / "icl_confusion_supersense.png")
errors = preds[~preds["correct"]]
errors.groupby(["gold_supersense", "pred_supersense"]).size().sort_values(ascending=False).head(15).to_frame("n")

## 3. Are errors near misses?

An error is *within hypersense* when the predicted supersense has the same hypersense as the gold one
(e.g. *act* → *event*, both *dynamic_situation*).

In [ ]:
same_hyper = errors["pred_supersense"].map(SUPER2HYPER) == errors["gold_hypersense"]
print(f"{len(errors)} errors: {same_hyper.mean():.0%} within the gold hypersense, "
      f"{(~same_hyper).mean():.0%} across hypersenses")
print(f"Supersense accuracy {preds['correct'].mean():.3f}  vs  hypersense accuracy "
      f"{(preds['gold_hypersense'] == preds['pred_hypersense']).mean():.3f}")

## 4. Which inputs are hard? Error rate by property of the definition

In [ ]:
# Same shallow cues as in 00_data_exploration.
props = preds.assign(
    n_words=preds["definition"].str.split().str.len(),
    demonym=preds["definition"].str.match(r"Habitant|Habitante", flags=re.I),
    synonym_or_variant=preds["definition"].str.match(r"(Synonyme|Variante|Autre nom|Pluriel) de", flags=re.I),
    complex_type=preds["gold_supersense"].str.contains("*", regex=False),
)
props["length"] = pd.cut(props["n_words"], [0, 3, 7, 15, 1000], labels=["1-3", "4-7", "8-15", ">15"])
rows = [(name, value, len(group), 1 - group["correct"].mean())
        for name in ["length", "demonym", "synonym_or_variant", "complex_type"]
        for value, group in props.groupby(name, observed=True)]
pd.DataFrame(rows, columns=["property", "value", "n", "error_rate"]).round(3)

## 5. What does the lemma change?

Senses classified correctly only with the lemma, and only without it, in the two runs of the lemma comparison.

In [ ]:
# The configuration of the lemma comparison (stage 4 of 01_icl): the only pair of runs that differ by the lemma.
LEMMA_ID = next(c for c in icl_dev["config_id"] if c.endswith("_lemma")
                and c.replace("_lemma", "_nolemma") in set(icl_dev["config_id"]))
both = load_preds(LEMMA_ID).merge(load_preds(LEMMA_ID.replace("_lemma", "_nolemma"))[
    ["sense_id", "pred_supersense", "correct"]], on="sense_id", suffixes=("", "_nolemma"))
helped = both[both["correct"] & ~both["correct_nolemma"]]
hurt = both[~both["correct"] & both["correct_nolemma"]]
print(f"{LEMMA_ID}: the lemma fixes {len(helped)} senses and breaks {len(hurt)}")
display(helped[["lemma", "definition", "gold_supersense", "pred_supersense_nolemma"]].head(10))
display(hurt[["lemma", "definition", "gold_supersense", "pred_supersense"]].head(10))

## 6. Errors to read by hand

In [ ]:
pd.set_option("display.max_colwidth", 90)
errors.sample(min(30, len(errors)), random_state=42)[["lemma", "definition", "gold_supersense", "pred_supersense", "raw_output"]]

## 7. Test sets

In [ ]:
for split in ["rand-test", "freq-test"]:
    test_ids = metrics[(metrics["method"] == "icl") & (metrics["split"] == split)]["config_id"]
    if len(test_ids):
        test = load_preds(test_ids.iloc[-1], split)
        s = evaluate(test["gold_supersense"].tolist(), test["pred_supersense"].tolist(), SUPERSENSES)
        worst = sorted(s["per_class"].items(), key=lambda kv: kv[1])[:5]
        print(f"{split}: macro-F1 {s['macro_f1']:.3f}, accuracy {s['accuracy']:.3f}; worst classes {worst}")
        plot_confusion(test["gold_supersense"].tolist(), test["pred_supersense"].tolist(), SUPERSENSES,
                       FIG_DIR / f"icl_confusion_{split}.png")

## Observations

- (to be written after reading the errors)